In [ ]:
import numpy as np
from scipy.stats import chi2
%matplotlib widget
import matplotlib.pyplot as plt
import mtt

# Settings

In [ ]:
# Map
n = 100
q = .2
r = 1.
avg_targets = 16
PS = .99
radius = 250.
model = mtt.Model.CVM(r=r, q=q)

rng = mtt.get_rng(seed=42)
radar_seed = rng.integers(0, 2**31-1)

sim_birth_int = (1 - PS) * avg_targets
tmap = mtt.scenario_random_spawn(
    ndat=n, model=model, lambda_births=sim_birth_int, lambda_init_births=avg_targets, radius=radius, PS=PS, init_vel_var=1., rng=rng
)

# Radar
lambd = 1e-4
PD = .9
radar = mtt.Radar(lambd=lambd, det_prob=PD, radius=radius, gate_by_rad=True, seed=radar_seed)

# Tracker
PG = .9999
min_hypot_w = 1e-5
min_r = 1e-5
min_w = 1e-7
max_hypots = 1000
conf_thr = 0.4
recyclate = True

n_std = chi2.ppf(PG, df=2) if PG < 1. else 20

# One Gaussian Spawning
dim = model.state_dim
gm = mtt.GaussianMixture(dim)
gm0 = mtt.GaussianMixture(dim)
assert dim == 4

pos_var = (1.1 * 2 * radius)**2
vel_var = 1.
P = np.diag([pos_var, pos_var, vel_var, vel_var])
M = np.zeros(shape=(4,), dtype=float)
gm.push(5e-3, M, P)
gm0.push(0.75*avg_targets, M, P)

# Grid Spawning
gm_grid = mtt.GaussianMixture(dim=4)
gm0_grid = mtt.GaussianMixture(dim=4)

spacing = 100
pos_var = spacing**2
coords = np.arange(-radius, radius + spacing, spacing)
birth_w = 5e-4
birth_w0 = 0.75*avg_targets / len(coords)**2 / 0.8
P = np.diag([pos_var, pos_var, vel_var, vel_var]).astype(np.float64)

for x in coords:
    for y in coords:
        if np.hypot(x, y) <= radius * 1.1:
            M = np.array([x, y, 0.0, 0.0], dtype=np.float64)
            gm_grid.push(birth_w, M, P)
            gm0_grid.push(birth_w0, M, P)

# PMBM

In [ ]:
pmbm_tracker = mtt.PmbmTracker(birth_mixture=gm_grid, F=model.F, Q=model.Q, H=model.H, R=model.R, PD=PD, PS=PS, clutter_int=lambd, PG=PG, min_global_hypot_weight=min_hypot_w,
                               min_bernoulli_exist_prob=min_r, min_poisson_weight=min_w, max_hypothesis=max_hypots, recyclate=recyclate, conf_thr=conf_thr, sparsify=False,
                               birth_mixture0=gm0_grid
)

fig, ax = plt.subplots(figsize=(6, 6))
plotter = mtt.Plotter(ax, params={
        'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std},
        'ucov_ellipse': {'alpha':0.5, 'facecolor':'green', 'edgecolor':'green', 'label':'unconf. cov. ell.', 'n_std':n_std},
        'traj_arrow': {'closed':True, 'facecolor':'black', 'edgecolor':'none', 'zorder':10, 'arrow_scale':4.},
        },
        config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
)
# plotter.plot_airports(inner_airports=[], edge_airports=[])

# mtt.make_gif(fig, tmap, radar, pmbm_tracker, plotter, "gifs/pmbm.gif")
sim = mtt.InteractiveSimulator(fig, tmap, radar, pmbm_tracker, plotter, 'PMBM')

# PMBM - merging components

In [ ]:
pmbm_merge_tracker = mtt.PmbmTracker(birth_mixture=gm_grid, F=model.F, Q=model.Q, H=model.H, R=model.R, PD=PD, PS=PS, clutter_int=lambd, PG=PG, min_global_hypot_weight=min_hypot_w,
                                     min_bernoulli_exist_prob=min_r, min_poisson_weight=min_w, max_hypothesis=max_hypots, recyclate=recyclate, conf_thr=conf_thr, merge_comps=True,
                                     birth_mixture0=gm0_grid
)

fig, ax = plt.subplots(figsize=(6, 6))
plotter = mtt.Plotter(ax, params={
        'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std},
        'ucov_ellipse': {'alpha':0.5, 'facecolor':'green', 'edgecolor':'green', 'label':'unconf. cov. ell.', 'n_std':n_std}
        },
        config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
)
# plotter.plot_airports(inner_airports=[], edge_airports=[])

# mtt.make_gif(fig, tmap, radar, pmbm_merge_tracker, plotter, "gifs/pmbm_merge.gif")
sim = mtt.InteractiveSimulator(fig, tmap, radar, pmbm_merge_tracker, plotter, 'PMBM + inner merging')

# PMBM - more agressive merging

In [ ]:
pmbm_mergep_tracker = mtt.PmbmTracker(birth_mixture=gm_grid, F=model.F, Q=model.Q, H=model.H, R=model.R, PD=PD, PS=PS, clutter_int=lambd, PG=PG, min_global_hypot_weight=min_hypot_w,
                                     min_bernoulli_exist_prob=min_r, min_poisson_weight=min_w, max_hypothesis=max_hypots, recyclate=recyclate, conf_thr=conf_thr, merge_comps=True, merge_thr=4.,
                                     birth_mixture0=gm0_grid
)

fig, ax = plt.subplots(figsize=(6, 6))
plotter = mtt.Plotter(ax, params={
        'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std},
        'ucov_ellipse': {'alpha':0.5, 'facecolor':'green', 'edgecolor':'green', 'label':'unconf. cov. ell.', 'n_std':n_std}
        },
        config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
)
# plotter.plot_airports(inner_airports=[], edge_airports=[])

# mtt.make_gif(fig, tmap, radar, pmbm_mergep_tracker, plotter, "gifs/pmbm_merge_pp.gif")
sim = mtt.InteractiveSimulator(fig, tmap, radar, pmbm_mergep_tracker, plotter, 'PMBM')

# TOMB/P

In [ ]:
tomb_tracker = mtt.TombTracker(birth_mixture=gm_grid, F=model.F, Q=model.Q, H=model.H, R=model.R, PD=PD, PS=PS, clutter_int=lambd, PG=PG, min_global_hypot_weight=min_hypot_w,
                               min_bernoulli_exist_prob=min_r, min_poisson_weight=min_w, max_hypothesis=max_hypots, recyclate=recyclate, conf_thr=conf_thr,
                               birth_mixture0=gm0_grid
)

fig, ax = plt.subplots(figsize=(6, 6))
plotter = mtt.Plotter(ax, params={
        'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std}
        },
        config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
)
# plotter.plot_airports(inner_airports=[], edge_airports=[])

# mtt.make_gif(fig, tmap, radar, tracker, plotter, "gifs/tomb.gif")
sim = mtt.InteractiveSimulator(fig, tmap, radar, tomb_tracker, plotter, 'TOMB/P')

# MOMB/P

In [ ]:
momb_tracker = mtt.MombTracker(birth_mixture=gm_grid, F=model.F, Q=model.Q, H=model.H, R=model.R, PD=0.9, PS=PS, clutter_int=lambd, PG=PG, alpha=0.0, min_global_hypot_weight=min_hypot_w,
                               min_bernoulli_exist_prob=min_r, min_poisson_weight=min_w, max_hypothesis=max_hypots, recyclate=recyclate, conf_thr=conf_thr, birth_mixture0=gm0_grid
)

fig, ax = plt.subplots(figsize=(6, 6))
plotter = mtt.Plotter(ax, params={
        'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std}
        },
        config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
)
# plotter.plot_airports(inner_airports=[], edge_airports=[])

# mtt.make_gif(fig, tmap, radar, tracker, plotter, "gifs/momb.gif")
sim = mtt.InteractiveSimulator(fig, tmap, radar, momb_tracker, plotter, 'MOMB/P')

# IMOMB/P

In [ ]:
imomb_tracker = mtt.MombTracker(birth_mixture=gm_grid, F=model.F, Q=model.Q, H=model.H, R=model.R, PD=0.9, PS=PS, clutter_int=lambd, PG=PG, alpha=0.5, min_global_hypot_weight=min_hypot_w,
                               min_bernoulli_exist_prob=min_r, min_poisson_weight=min_w, max_hypothesis=max_hypots, recyclate=recyclate, conf_thr=conf_thr, birth_mixture0=gm0_grid
)

fig, ax = plt.subplots(figsize=(6, 6))
plotter = mtt.Plotter(ax, params={
        'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std}
        },
        config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
)
# plotter.plot_airports(inner_airports=[], edge_airports=[])

# mtt.make_gif(fig, tmap, radar, imomb_tracker, plotter, "gifs/imomb.gif")
sim = mtt.InteractiveSimulator(fig, tmap, radar, imomb_tracker, plotter, 'IMOMB/P')

# PHD

In [ ]:
phd_tracker = mtt.PhdTracker(birth_mixture=gm_grid, F=model.F, Q=model.Q, H=model.H, R=model.R, PD=PD, PS=PS, clutter_int=lambd, PG=PG,
                             trunc_thr=min_w, merge_thr=4, max_components=max_hypots, conf_thr=conf_thr, birth_mixture0=gm0_grid
)

fig, ax = plt.subplots(figsize=(6, 6))
plotter = mtt.Plotter(ax, params={
        'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
        'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std},
        'ucov_ellipse': {'alpha':0.5, 'facecolor':'green', 'edgecolor':'green', 'label':'unconf. cov. ell.', 'n_std':n_std},
        'traj_arrow': {'closed':True, 'facecolor':'black', 'edgecolor':'none', 'zorder':10, 'arrow_scale':4.},
        },
        config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
)
# plotter.plot_airports(inner_airports=inner_airports, edge_airports=edge_airports)

# mtt.make_gif(fig, tmap, radar, tracker, plotter, "gifs/phd.gif")
sim = mtt.InteractiveSimulator(fig, tmap, radar, phd_tracker, plotter, 'PHD')

# Comparison

In [ ]:
# Empty for just plotting the traj map
n = tmap.size
mock_e = [mtt.MockTracker((np.empty((0, dim), dtype=float), np.empty((0, dim, dim), dtype=float)), None) for _ in range(n)]

fig, axes = plt.subplots(2, 3, figsize=(14, 14))
ax1, ax2, ax3, ax4, ax5, ax6 = axes.flatten()
plotter_params = {
    'inner_airport': {'alpha':0.5, 'facecolor': 'yellow', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
    'edge_airport': {'alpha':0.5, 'facecolor': 'none', 'edgecolor':'black', 'linestyle':'--', 'n_std':n_std},
    'cov_ellipse': {'alpha':0.5, 'facecolor':'orange', 'edgecolor':'red', 'label':'cov. ell.', 'n_std':n_std},
    'ucov_ellipse': {'alpha':0.5, 'facecolor':'green', 'edgecolor':'green', 'label':'unconf. cov. ell.', 'n_std':n_std}
}
plotter_config = {'uest_traj': False, 'ucov_ellipse': False, 'est_traj': False, 'target': False, 'traj_arrow': False}
plotter_config2 = {'uest_traj': False, 'ucov_ellipse': True, 'est_traj': False, 'target': False, 'traj_arrow': False}

plotter1 = mtt.Plotter(ax1, params=plotter_params, config=plotter_config)
plotter2 = mtt.Plotter(ax2, params=plotter_params, config=plotter_config)
plotter3 = mtt.Plotter(ax3, params=plotter_params, config=plotter_config)
plotter4 = mtt.Plotter(ax4, params=plotter_params, config=plotter_config)
plotter5 = mtt.Plotter(ax5, params=plotter_params, config=plotter_config)
plotter6 = mtt.Plotter(ax6, params=plotter_params, config=plotter_config)

plotters = [plotter1, plotter2, plotter3, plotter4, plotter5, plotter6]
estimators = [2, 2, 2, 1, 1, 1]

pmbm_tracker.reset()
pmbm_merge_tracker.reset()
pmbm_mergep_tracker.reset()
tomb_tracker.reset()
momb_tracker.reset()
imomb_tracker.reset()
phd_tracker.reset()

sim_multi = mtt.InteractiveSimulator(
    fig=fig, 
    tmap=tmap, 
    radar=radar,
    trackers=[pmbm_tracker, pmbm_merge_tracker, pmbm_mergep_tracker, tomb_tracker, imomb_tracker, phd_tracker],
    plotters=plotters,
    labels=['PMBM', 'PMBM + merging', 'PMBM + mergin++', 'TOMB/P', 'IMOMB/P', 'PHD'],
    calc_gospa=True,
    gospa_c = 10.,
    gospa_p = 2.
)

sim_multi.show()

In [ ]:
sim_multi.plot_gospa(rms=False, cols=3, ignore_t0=True);